# Linear Regression from Scratch

Building linear regression from the ground up. Understanding gradient descent, loss functions, and optimisation.

## The Model: ŷ = w·x + b

Find weights w and bias b that minimise the difference between predictions ŷ and true values y.

Simple but powerful — basis for neural networks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## Linear Regression Class

Implementing linear regression with both normal equation and gradient descent.

In [ ]:
class LinearRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.cost_history = []
    
    def fit_normal_equation(self, X, y):
        """Fit using closed-form normal equation"""
        # Add bias term (column of ones)
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        
        # Normal equation: w = (X^T * X)^(-1) * X^T * y
        self.weights = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
        self.bias = self.weights[0]
        self.weights = self.weights[1:]
        
        return self
    
    def fit_gradient_descent(self, X, y):
        """Fit using gradient descent"""
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for i in range(self.n_iterations):
            # Forward pass: predictions
            y_pred = self._predict(X)
            
            # Compute gradients
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))  # ∂L/∂w
            db = (1/n_samples) * np.sum(y_pred - y)         # ∂L/∂b
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Track cost
            cost = self._compute_cost(X, y)
            self.cost_history.append(cost)
            
            # Print progress
            if i % 100 == 0:
                print(f"Iteration {i}: Cost = {cost:.6f}")
        
        return self
    
    def _predict(self, X):
        """Make predictions"""
        return np.dot(X, self.weights) + self.bias
    
    def _compute_cost(self, X, y):
        """Compute MSE cost"""
        n_samples = len(y)
        y_pred = self._predict(X)
        return (1/(2*n_samples)) * np.sum((y_pred - y)**2)
    
    def predict(self, X):
        """Public predict method"""
        return self._predict(X)
    
    def get_coefficients(self):
        """Return learned parameters"""
        return self.weights, self.bias

## Normal Equation

Closed-form solution: w = (XᵀX)⁻¹Xᵀy. No iteration needed. Works for small datasets.

In [ ]:
# Generate data
X, y = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit with normal equation
model_ne = LinearRegressionScratch()
model_ne.fit_normal_equation(X_train, y_train)

# Predictions
y_pred_ne = model_ne.predict(X_test)

# Results
weights_ne, bias_ne = model_ne.get_coefficients()
print(f"Normal Equation - Weights: {weights_ne}, Bias: {bias_ne:.3f}")
print(f"MSE: {mean_squared_error(y_test, y_pred_ne):.3f}")
print(f"R²: {r2_score(y_test, y_pred_ne):.3f}")

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, alpha=0.7, label='Test data')
plt.plot(X_test, y_pred_ne, color='red', linewidth=2, label='Normal equation fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Linear Regression: Normal Equation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Gradient Descent

Iteratively update weights: w = w - α·∂L/∂w. Learning rate α controls step size.

In [ ]:
# Fit with gradient descent
model_gd = LinearRegressionScratch(learning_rate=0.1, n_iterations=1000)
model_gd.fit_gradient_descent(X_train, y_train)

# Predictions
y_pred_gd = model_gd.predict(X_test)

# Results
weights_gd, bias_gd = model_gd.get_coefficients()
print(f"Gradient Descent - Weights: {weights_gd}, Bias: {bias_gd:.3f}")
print(f"MSE: {mean_squared_error(y_test, y_pred_gd):.3f}")
print(f"R²: {r2_score(y_test, y_pred_gd):.3f}")

# Plot cost history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(model_gd.cost_history)
plt.xlabel('Iteration')
plt.ylabel('Cost (MSE)')
plt.title('Gradient Descent: Cost vs Iterations')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_test, y_test, alpha=0.7, label='Test data')
plt.plot(X_test, y_pred_gd, color='green', linewidth=2, label='Gradient descent fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Linear Regression: Gradient Descent')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare with normal equation
print(f"\nComparison:")
print(f"Normal Equation: w={weights_ne[0]:.3f}, b={bias_ne:.3f}")
print(f"Gradient Descent: w={weights_gd[0]:.3f}, b={bias_gd:.3f}")
print(f"Difference: w={abs(weights_ne[0]-weights_gd[0]):.6f}, b={abs(bias_ne-bias_gd):.6f}")

## Learning Rate Impact

Too small = slow convergence. Too large = overshoot minimum.

In [ ]:
# Test different learning rates
learning_rates = [0.001, 0.01, 0.1, 0.5]
colors = ['blue', 'green', 'red', 'orange']

plt.figure(figsize=(12, 8))

for lr, color in zip(learning_rates, colors):
    model = LinearRegressionScratch(learning_rate=lr, n_iterations=200)
    model.fit_gradient_descent(X_train, y_train)
    
    plt.plot(model.cost_history, color=color, linewidth=2, 
             label=f'LR = {lr}')

plt.xlabel('Iteration')
plt.ylabel('Cost (MSE)')
plt.title('Gradient Descent: Learning Rate Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')  # Log scale for better visualization
plt.show()

print("Learning Rate Guidelines:")
print("- Too small (0.001): Very slow convergence")
print("- Good (0.01-0.1): Steady decrease")
print("- Too large (0.5): May overshoot or diverge")

## Feature Scaling

Important for convergence. Standardisation: mean=0, std=1.

In [ ]:
# Generate data with different scales
X_scaled = np.random.randn(100, 2)
X_scaled[:, 1] = X_scaled[:, 1] * 100 + 50  # Different scale
y_scaled = X_scaled[:, 0] * 2 + X_scaled[:, 1] * 0.01 + np.random.randn(100) * 0.1

# Without scaling
model_no_scale = LinearRegressionScratch(learning_rate=0.01, n_iterations=500)
model_no_scale.fit_gradient_descent(X_scaled, y_scaled)

# With scaling
X_scaled_norm = (X_scaled - X_scaled.mean(axis=0)) / X_scaled.std(axis=0)
model_scaled = LinearRegressionScratch(learning_rate=0.1, n_iterations=500)
model_scaled.fit_gradient_descent(X_scaled_norm, y_scaled)

# Compare convergence
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(model_no_scale.cost_history, label='No scaling', color='red')
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Without Feature Scaling')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(model_scaled.cost_history, label='With scaling', color='green')
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('With Feature Scaling')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final cost without scaling: {model_no_scale.cost_history[-1]:.6f}")
print(f"Final cost with scaling: {model_scaled.cost_history[-1]:.6f}")
print("\nScaling allows higher learning rate and faster convergence!")

## Regularisation

L2 regularisation (Ridge) prevents overfitting by penalising large weights.

In [ ]:
class LinearRegressionRegularised:
    def __init__(self, learning_rate=0.01, n_iterations=1000, lambda_reg=0.1):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.lambda_reg = lambda_reg  # Regularisation strength
        self.weights = None
        self.bias = None
        self.cost_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for i in range(self.n_iterations):
            y_pred = self._predict(X)
            
            # Gradients with L2 regularisation
            dw = (1/n_samples) * (np.dot(X.T, (y_pred - y)) + self.lambda_reg * self.weights)
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Update
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Cost with regularisation
            cost = self._compute_cost(X, y)
            self.cost_history.append(cost)
        
        return self
    
    def _predict(self, X):
        return np.dot(X, self.weights) + self.bias
    
    def _compute_cost(self, X, y):
        n_samples = len(y)
        y_pred = self._predict(X)
        mse = (1/(2*n_samples)) * np.sum((y_pred - y)**2)
        reg_term = (self.lambda_reg/(2*n_samples)) * np.sum(self.weights**2)
        return mse + reg_term
    
    def predict(self, X):
        return self._predict(X)

# Test regularisation
lambdas = [0, 0.01, 0.1, 1.0]
colors = ['red', 'orange', 'green', 'blue']

plt.figure(figsize=(12, 8))

for lambda_reg, color in zip(lambdas, colors):
    model = LinearRegressionRegularised(learning_rate=0.1, lambda_reg=lambda_reg)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    
    plt.subplot(2, 2, 1)
    plt.plot(model.cost_history, color=color, label=f'λ={lambda_reg}')
    
    plt.subplot(2, 2, 2)
    plt.scatter(X_test, y_test, alpha=0.5, color='gray')
    plt.plot(X_test, y_pred, color=color, linewidth=2, label=f'λ={lambda_reg}, MSE={mse:.2f}')

plt.subplot(2, 2, 1)
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Regularisation: Cost vs Iterations')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Regularisation: Different λ Values')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Regularisation:")
print("- λ=0: No regularisation (may overfit)")
print("- λ>0: Penalises large weights (prevents overfitting)")
print("- Higher λ = stronger regularisation = smaller weights")

## Key Takeaways

- **Linear regression** assumes linear relationship. Check with scatter plots.
- **Gradient descent**: compute gradients, update parameters, repeat.
- **Feature scaling** important for convergence. Standardise inputs.
- **Regularisation** (L1/L2) prevents overfitting by penalising large weights.
- **Linear models** interpretable — coefficients show feature importance.

## Exercises

1. **Gradient Calculation**: Derive ∂L/∂w for MSE loss. Show your work.

2. **Learning Rate**: You notice your cost is oscillating during training. What should you do?

3. **Convergence**: How do you know when gradient descent has converged?

4. **Regularisation**: When should you use L1 vs L2 regularisation?

5. **Implementation**: Implement stochastic gradient descent (one sample at a time) instead of batch GD.

## Solutions

1. **Gradient Calculation**: L = (1/2n)Σ(y - ŷ)², ŷ = w·x + b. ∂L/∂w = (1/n)Σ x·(ŷ - y) = (1/n)Xᵀ·(ŷ - y)

2. **Learning Rate**: Reduce the learning rate. Oscillating means the step size is too large, overshooting the minimum.

3. **Convergence**: Cost stops decreasing significantly, or gradient magnitude becomes very small, or no improvement over several iterations.

4. **Regularisation**: L1 (Lasso) for feature selection (creates sparse models). L2 (Ridge) for preventing overfitting when all features are useful.

5. **Implementation**: In the gradient calculation, use one random sample instead of all samples. This is faster but noisier.